# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [1]:
import sys
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")
print("Versión de Python:", sys.version)
print("Ruta de Python:", sys.executable)

Versión de Python: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Ruta de Python: c:\Users\marcg\TinyBird\venv\Scripts\python.exe


In [2]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

import sys
print(sys.executable)

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

c:\Users\marcg\TinyBird\venv\Scripts\python.exe


## Data

In [3]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0379


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
0,r_040cad9cfeba52e7,2026-11-13 03:58:14.135000000,u_a25e9888,"{'device': 'android', 'page': 'category', 'reg...","[{'product_id': 'p_5c64c5', 'ts': '2026-11-13 ...","[p_032d24, p_932093, p_adc3e6, p_10e9e4, p_7e8...","[0.032807, 0.004071, 0.278884, 0.000084, 0.003...",NaN,NaN
1,r_9026deb0a586c68c,2026-11-08 23:59:04.740000000,u_405453fd,"{'device': 'ios', 'page': 'product', 'region':...","[{'product_id': 'p_fdb0b1', 'ts': '2026-11-08 ...","[p_b89c67, p_0ceafe, p_adc3e6, p_e95fee, p_dc3...","[0.000755, 0.002774, 0.269587, 0.01579, 0.0106...",NaN,NaN
2,r_3fd8c9c92815ddf7,2026-11-07 21:47:46.885000000,u_e5eb64ee,"{'device': 'ios', 'page': 'home', 'region': 'M...",[],"[p_d72d81, p_4077a5, p_92d45d, p_2c88d5, p_7a1...","[0.014146, 0.007666, 0.03011, 0.233633, 0.0089...",NaN,NaN


In [4]:
# Co-views computed inside RawTree: for every product, the 20 products most viewed in the same sessions
coviews = sql(f"""
    SELECT a.product_id AS p, b.product_id AS q, count() AS n
    FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS a
    INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS b
      ON a.session_id = b.session_id
    WHERE a.product_id != b.product_id
    GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
covisit = coviews.groupby("p").q.apply(list).to_dict()
print(len(covisit), "products with co-views")

5000 products with co-views


In [5]:
# c04: importar nuestro recommender con artefactos de corte
import os, sys
sys.path.insert(0, ".")
sys.modules.pop("recommender4", None)     # Evita versiones cacheadas anteriores
import recommender4 as ours

assert ours.LGB_MODEL is not None, "No se cargó el modelo de 22 features"

# Ranking de los más vendidos ANTES del corte (baseline real)
best = sql(f"""
    SELECT product_id::String AS product_id, count() AS n
    FROM shop.history_events
    WHERE event_type = 'purchase' AND ts::DateTime64(3) < toDateTime('{CUT}')
    GROUP BY product_id ORDER BY n DESC LIMIT 200""").product_id.tolist()

print("¡Modelo v8 de 22 features cargado con éxito en el notebook!")

¡Modelo v8 de 22 features cargado con éxito en el notebook!


## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [6]:
# c07: políticas
def baseline(row):
    """Los 10 más vendidos antes del corte."""
    return best[:10]

def coview(row):
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views or [])):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

def lgbm(row):
    # Aquí pasamos los nuevos campos para activar el Cold Start de usuario y contexto
    return ours.recommend(
        row.recent_views,
        best,
        user_id=str(getattr(row, "user_id", "") or ""),
        context=getattr(row, "context", None),
    )

POLICIES = {"baseline": baseline, "coview": coview, "lgbm": lgbm}

## Evaluation

In [7]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0113


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.394083,0.007073,0.047006,4042.0
coview,0.235077,0.007992,0.010997,5565.0
lgbm,0.836767,0.060542,0.131629,7913.0


In [8]:
# Comprobación estricta de novedad (excluyendo lo que ya vio en recent_views)
def evaluate_novelty(policy, df):
    hits_total = 0
    hits_novel = 0
    count_clicks = 0
    
    for row in df.itertuples():
        if pd.isna(row.clicked):
            continue
        count_clicks += 1
        
        recs = policy(row)
        clicked = row.clicked
        
        # Extraemos los productos vistos en la sesión actual
        viewed_ids = [v["product_id"] for v in (row.recent_views or []) if isinstance(v, dict)]
        
        is_hit = clicked in recs
        is_novel = clicked not in viewed_ids
        
        if is_hit:
            hits_total += 1
            if is_novel:
                hits_novel += 1
                
    print(f"Clicks analizados: {count_clicks}")
    print(f"Hit Rate Global: {hits_total / max(count_clicks, 1):.4f}")
    print(f"Hit Rate de productos NUEVOS (no vistos en recent_views): {hits_novel / max(count_clicks, 1):.4f}")

evaluate_novelty(lgbm, test)

Clicks analizados: 3786
Hit Rate Global: 0.8368
Hit Rate de productos NUEVOS (no vistos en recent_views): 0.7467


## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.